# Build Capri AI APK
This notebook compiles the Capri Android project for `arm64-v8a`. It uses CPU inference so it does not require a Colab GPU.

Upload `CapriAI_Android.zip` when prompted. The GGUF model is imported later inside the Android app; it is not embedded in the APK.

In [ ]:
from google.colab import files
uploaded = files.upload()  # choose CapriAI_Android.zip
zip_name = next(name for name in uploaded if name.lower().endswith('.zip'))
print('Uploaded:', zip_name)

In [ ]:
!apt-get update -qq
!apt-get install -y -qq openjdk-21-jdk git wget unzip cmake ninja-build
import os, zipfile
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content')
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']
%cd /content/CapriAI_Android
!java -version

In [ ]:
# Fetch the Android GGUF runtime source and its pinned native dependencies.
!git clone --depth 1 https://github.com/tensai-app/llama.kt.git libs/llama.kt
%cd /content/CapriAI_Android/libs/llama.kt
!git submodule update --init --recursive
!bash scripts/bootstrap.sh!bash scripts/build-opencl.sh


In [ ]:
# Install Android command-line tools, platform 35, build tools and NDK.
import os, pathlib, subprocess
sdk = '/content/android-sdk'
os.makedirs(sdk + '/cmdline-tools', exist_ok=True)
!wget -q https://dl.google.com/android/repository/commandlinetools-linux-13114758_latest.zip -O /content/cmdline-tools.zip
!mkdir -p /content/cmdline-tools-tmp && unzip -q -o /content/cmdline-tools.zip -d /content/cmdline-tools-tmp
!mkdir -p /content/android-sdk/cmdline-tools/latest
!cp -r /content/cmdline-tools-tmp/cmdline-tools/* /content/android-sdk/cmdline-tools/latest/
os.environ['ANDROID_HOME'] = sdk
os.environ['ANDROID_SDK_ROOT'] = sdk
os.environ['PATH'] += ':' + sdk + '/cmdline-tools/latest/bin:' + sdk + '/platform-tools'
!yes | sdkmanager --licenses >/dev/null
!sdkmanager 'platform-tools' 'platforms;android-35' 'build-tools;35.0.0' 'ndk;27.2.12479018'

In [ ]:
# Download Gradle 9.5.0 and build a debug APK.
!wget -q https://services.gradle.org/distributions/gradle-9.5-bin.zip -O /content/gradle.zip
!unzip -q -o /content/gradle.zip -d /content
%cd /content/CapriAI_Android
!export ANDROID_HOME=/content/android-sdk ANDROID_SDK_ROOT=/content/android-sdk ANDROID_NDK_HOME=/content/android-sdk/ndk/27.2.12479018; /content/gradle-9.5/bin/gradle --no-daemon :app:assembleDebug

In [ ]:
apk = '/content/CapriAI_Android/app/build/outputs/apk/debug/app-debug.apk'
assert os.path.exists(apk), 'APK not found. Check the Gradle error output above.'
print('APK ready:', apk, 'size MB:', round(os.path.getsize(apk)/1024/1024, 1))
files.download(apk)